<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 1 · Python Programming Foundations for Data Science
### Notebook 04 · Text Methods and Titanic
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
Chapter 4 uses Titanic as a string-heavy warmup. The point is not analysis yet. The
point is to get comfortable with raw text, cleaning, and counting.

### Goals
- Open a local Titanic CSV or download a copy if needed.
- Inspect the raw rows as text.
- Clean a few names and count embarkation ports.

## Setup
We start by making the book root explicit so the Titanic file can be found with
stable relative paths.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "1_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


## Find and Preview the Raw CSV
Before we clean anything, we locate the file and inspect the rows as raw text.
That keeps the later cleaning grounded in what actually arrived.


In [ ]:
import csv
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = 'https://hilpisch.com/Titanic.csv'
LOCAL_CANDIDATES = [
    BOOK_ROOT / 'data' / 'Titanic.csv',
    Path('../2_data/data') / 'Titanic.csv',
    Path('../../2_data/data') / 'Titanic.csv',
]

def resolve_data_path() -> Path:
    for candidate in LOCAL_CANDIDATES:
        if candidate.exists():
            return candidate
    target = LOCAL_CANDIDATES[0]
    target.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(DATA_URL, target)
    return target

def preview_rows(path: Path, limit: int = 5) -> list[str]:
    rows = []
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.reader(handle)
        for _ in range(limit):
            try:
                row = next(reader)
            except StopIteration:
                break
            rows.append(', '.join(row))
    return rows


## Clean Names and Count Ports
The next step stays simple on purpose: tidy a few names and count embarkation
ports. That is enough to turn text handling into a small data task.


In [ ]:
def clean_name(name: str) -> str:
    return ' '.join(name.strip().split()).title()

def count_embarked(path: Path) -> dict[str, int]:
    counts = {}
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            port = (row.get('Embarked') or '').strip().upper() or '?'
            counts[port] = counts.get(port, 0) + 1
    return counts

data_path = resolve_data_path()
raw_preview = preview_rows(data_path)
counts = count_embarked(data_path)

print(f'Using Titanic CSV at {data_path.resolve()}')
print('Raw preview:')
for line in raw_preview:
    print(f'  {line}')
print('Embarkation counts:')
for port, count in sorted(counts.items()):
    print(f'  {port}: {count}')
print('Example cleaned names:')
with data_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    for index, row in enumerate(reader):
        print(f"  {clean_name(row.get('Name') or '')}")
        if index >= 4:
            break


## Save a Reusable Summary
The final cell writes a compact text summary so the cleaned preview and counts
can be revisited without rerunning the whole notebook.


In [ ]:
from pathlib import Path

artifact_path = BOOK_ROOT / 'artifacts' / 'titanic_strings_summary.txt'
artifact_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_names = []
with data_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    for index, row in enumerate(reader):
        cleaned_names.append(clean_name(row.get('Name') or ''))
        if index >= 9:
            break

lines = ['Titanic strings summary', '', 'Raw preview:']
lines.extend(f'  {line}' for line in raw_preview)
lines.extend(['', 'Embarkation counts:'])
lines.extend(f'  {port}: {count}' for port, count in sorted(counts.items()))
lines.extend(['', 'Cleaned names:'])
lines.extend(f'  {name}' for name in cleaned_names)
artifact_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(artifact_path.resolve())


### Reflection
- Why does the raw CSV preview matter before any analysis?
- What is the difference between text cleaning and data analysis?
- Which string operation would you reuse in a later module?

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">